# Solutions · Chapter 05-05 · Residuals

Worked answers to every exercise in `notebooks/05_regression/05-05_residuals.ipynb`.

Read the exercise, attempt it, and only then read on.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# SYNTHETIC: the chapter's four scenarios, all calibrated to RMSE 2.4000.
rng = np.random.default_rng(41)
n_rows = 200
x = rng.uniform(0, 10, n_rows)
noise = rng.normal(0, 1, n_rows)
group = rng.integers(0, 2, n_rows)

scenarios = {
    "A  plain noise": 3 + 2 * x + 2.4875 * noise,
    "B  curvature":   3 + 2 * x + 0.30 * (x - 5) ** 2 + 1.4403 * noise,
    "C  fanning":     3 + 2 * x + noise * (0.15 + 0.3854 * x),
    "D  two groups":  3 + 2 * x + 4.5 * (group - 0.5) + 0.8437 * noise,
}
fits = {}
for name, y in scenarios.items():
    model = LinearRegression().fit(x.reshape(-1, 1), y)
    fitted = model.predict(x.reshape(-1, 1))
    fits[name] = {"y": y, "fitted": fitted, "residual": y - fitted}

# SYNTHETIC: the chapter's 400 deliveries. TRUTH: minutes are quadratic in stops.
delivery_rng = np.random.default_rng(6)
n_deliveries = 400
distance_km = delivery_rng.uniform(1, 20, n_deliveries)
stops = delivery_rng.uniform(1, 12, n_deliveries)
minutes = (8 + 1.9 * distance_km + 0.55 * stops ** 2
           + delivery_rng.normal(0, 3.0, n_deliveries))
features = pd.DataFrame({"distance_km": distance_km, "stops": stops})

print("set up: four scenarios and %d deliveries" % n_deliveries)

## Quick understanding

### E1 · What a residual is, and why the sign matters

`residual = actual - prediction`, computed per row. Positive means the model under-predicted.

**Keeping the sign is the point because every metric throws it away.** MAE takes an absolute value and
MSE squares, so both map +5 and -5 to the same thing. That is correct for scoring - a miss is a miss -
and it is exactly what destroys the pattern. Residuals of +5, -5, +5, -5 and residuals of +5, +5, +5, +5
score identically and mean completely different things: the first is noise, the second is a model that is
wrong in one direction for every row it sees.

### E2 · Why corr(residual, fitted) is exactly zero

**Because if it were not, the fit would not be optimal** - any remaining linear relationship between the
residuals and the fitted values could be added to the predictions to reduce the squared error, so a
least-squares solution is by definition one where none is left.

The consequence is what makes the plot usable: it has a **known flat reference**. Structure you see in it
is real, and not an artefact of the plot.

### E3 · The four patterns

| Pattern | Action |
|---|---|
| Random cloud | Nothing - the model has taken what is available |
| A curve | Add a term for that feature, or use a model that bends |
| A fan | Consider a target transform, and report the varying spread |
| Bands or clumps | Find the missing column and add it |

## Hand calculation

### E4 · Five residuals

| actual | predicted | residual |
|---|---|---|
| 10 | 11 | **-1** |
| 12 | 13 | **-1** |
| 15 | 15 | **0** |
| 19 | 17 | **+2** |
| 24 | 19 | **+5** |

**The residuals rise steadily: -1, -1, 0, +2, +5.** A residual that increases monotonically with the
prediction means the model is missing a shape - the truth is bending upward faster than the straight line
does. This is scenario B with five rows.

Note that the MAE is 1.8, which sounds unremarkable. The pattern is the finding, not the size.

### E5 · A curvature test in two numbers

Mean of the first two residuals: (-1 + -1) / 2 = **-1.0**.
Mean of the last two: (2 + 5) / 2 = **+3.5**.

**Those two numbers are a curvature test because for a correct model they should both be about zero,
and any systematic difference between them means the mean residual depends on where you are in the
range.** That is exactly what "the mean residual changes with x" means, and it is the first table in the
chapter compressed into two numbers.

A caution worth carrying: with only five rows, -1.0 against +3.5 could be luck. The test is a *direction
to look*, not a verdict - which is why the chapter uses 200 rows.

### E6 · The correlation you expect

`corr(residual, actual) = sqrt(1 - R²) = sqrt(1 - 0.64) = sqrt(0.36) = **0.6**`.

No fitting required, and no property of the data involved. It is arithmetic that holds for every
least-squares fit.

### E7 · Alternating against blocked

In [ ]:
alternating = np.array([4, -4, 4, -4, 4, -4], dtype=float)
blocked = np.array([4, 4, 4, -4, -4, -4], dtype=float)

for label, e in [("alternating +-", alternating), ("blocked +++---", blocked)]:
    print("%-16s MAE %.4f   RMSE %.4f   mean %+.4f"
          % (label, np.abs(e).mean(), np.sqrt((e ** 2).mean()), e.mean()))

**Identical on every count: MAE 4, RMSE 4, mean 0.**

**I would rather have the alternating set**, because it is consistent with noise - the model is wrong by
4 in a direction that does not depend on anything, which may simply be the limit of what the data
supports.

The blocked set is a model that is wrong by +4 for the first half of its rows and -4 for the second half.
If those rows are ordered by anything at all - date, customer, size, the order the file was written -
that is structure, and structure is fixable. **The same total error, and one of the two is a bug.**

**Why the metrics cannot help:** both collapse the residuals into a single number by taking an absolute
value or a square, which discards both the sign and the ordering. Recovering either requires looking at
the residuals in the order they came, which is what the plot does.

## Coding

### E8 · A diagnostic function

In [ ]:
def diagnose(model, X, y):
    residual = np.asarray(y) - model.predict(X)
    overall_sd = residual.std()
    print("residual sd overall: %.4f\n" % overall_sd)
    for column in X.columns:
        parts = pd.qcut(X[column], 3, labels=["low", "mid", "high"], duplicates="drop")
        means = pd.Series(residual).groupby(parts.values, observed=True).mean()
        sds = pd.Series(residual).groupby(parts.values, observed=True).std()
        spread = means.max() - means.min()
        flag = "  <-- LOOK HERE" if spread > overall_sd else ""
        print("%-14s means %s   sds %s   range %.3f%s"
              % (column, np.round(means.to_numpy(), 3) + 0.0, np.round(sds.to_numpy(), 3),
                 spread, flag))


print("=== before adding stops_squared ===")
before = LinearRegression().fit(features, minutes)
diagnose(before, features, minutes)

print("\n=== after adding stops_squared ===")
extended = features.assign(stops_squared=features["stops"] ** 2)
after = LinearRegression().fit(extended, minutes)
diagnose(after, extended, minutes)

**Before, `stops` is flagged and `distance_km` is not.** The mean residual across the thirds of `stops`
ranges over 7.05 minutes against a residual standard deviation of 5.56 - the fault is larger than the
noise. After the fix nothing is flagged, and the residual sd has fallen to 3.00.

`stops_squared` appears in the second run and is not flagged either, which is expected rather than
reassuring: a *derived* column inherits its parent's ordering, so it reports the same range as `stops`
(0.201 for both) and is not independent evidence. The threshold of one standard deviation is a
convention, not a test - it is set so that a fault has to be comparable in size to the noise before it
interrupts you.

### E9 · The identity, on three very different fits

In [ ]:
identity_rng = np.random.default_rng(3)
n_check = 500
feature = identity_rng.uniform(0, 10, n_check)

checks = []
for noise_sd in [0.4, 3.0, 12.0]:
    target = 5 + 1.5 * feature + identity_rng.normal(0, noise_sd, n_check)
    model = LinearRegression().fit(feature.reshape(-1, 1), target)
    residual = target - model.predict(feature.reshape(-1, 1))
    r_squared = model.score(feature.reshape(-1, 1), target)
    checks.append({"noise sd": noise_sd, "R2": r_squared,
                   "corr(residual, actual)": np.corrcoef(residual, target)[0, 1],
                   "sqrt(1 - R2)": np.sqrt(1 - r_squared),
                   "corr(residual, fitted)": np.corrcoef(residual,
                                                         model.predict(feature.reshape(-1, 1)))[0, 1]})
print((pd.DataFrame(checks).round(6) + 0.0).to_string(index=False))

**Both identities hold to six decimal places at every noise level.**

The important reading is the direction: R-squared **0.9910** gives a residual-actual correlation of
**0.0948**, while R-squared **0.0676** gives **0.9656**. **The better the model, the flatter that plot
looks** - so a beginner using it as a diagnostic gets a signal that is not just meaningless but
backwards.

The last column is 0 to six decimal places every time, which is the flat reference the correct plot
relies on.

### E10 · Giving scenario D the group column

In [ ]:
d_target = scenarios["D  two groups"]
without = LinearRegression().fit(x.reshape(-1, 1), d_target)
with_group = LinearRegression().fit(np.column_stack([x, group]), d_target)

for label, model, design in [("without the group column", without, x.reshape(-1, 1)),
                             ("with the group column", with_group,
                              np.column_stack([x, group]))]:
    residual = d_target - model.predict(design)
    print("%-26s RMSE %.4f   R2 %.4f"
          % (label, np.sqrt((residual ** 2).mean()), model.score(design, d_target)))

print("\nthe group coefficient is %.4f; the data was built with a gap of 4.5"
      % with_group.coef_[1])
print("the noise the data was built with had sd 0.8437")

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.2), sharey=True)
with_residual = d_target - with_group.predict(np.column_stack([x, group]))

for ax, residual, title in [(left, fits["D  two groups"]["residual"], "Without the group column"),
                            (right, with_residual, "With it: the bands collapse onto zero")]:
    for value, marker, colour in [(0, "o", "#7B3294"), (1, "^", "#E69F00")]:
        keep = group == value
        ax.scatter(np.asarray(x)[keep], np.asarray(residual)[keep], s=18, alpha=0.7,
                   marker=marker, color=colour, label="group %d" % value)
    ax.axhline(0, color="#000000", linewidth=1.5)
    ax.set_xlabel("x")
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8.5)
left.set_ylabel("residual")

plt.tight_layout()
plt.show()

**RMSE falls from 2.4000 to 0.8140 - the error is cut to a third by one column.**

And 0.8140 is essentially the 0.8437 the data was built with, so the model is now as good as the data
allows. The fitted group coefficient is **4.5172** against a true gap of 4.5.

**This is why "bands or clumps" is the pattern to hope for.** Curvature needed a new term and got a
partial improvement; fanning could not be removed at all. A missing column, once found, is simply
restored, and the model becomes right.

**The uncomfortable part:** you can only add the column if you have it. In this chapter `group` was
sitting in the notebook the whole time. In real work the residual plot tells you a column exists and
nothing more - finding it is a conversation with whoever owns the data.

### E11 · Residuals against row index, sorted and unsorted

In [ ]:
sorted_order = np.argsort(features["stops"].to_numpy())
sorted_features = features.iloc[sorted_order].reset_index(drop=True)
sorted_minutes = minutes[sorted_order]

sorted_model = LinearRegression().fit(sorted_features, sorted_minutes)
sorted_residual = sorted_minutes - sorted_model.predict(sorted_features)

unsorted_model = LinearRegression().fit(features, minutes)
unsorted_residual = minutes - unsorted_model.predict(features)

print("sorted   RMSE %.4f   corr(residual, row index) %+.4f"
      % (np.sqrt((sorted_residual ** 2).mean()),
         np.corrcoef(sorted_residual, np.arange(len(sorted_residual)))[0, 1]))
print("unsorted RMSE %.4f   corr(residual, row index) %+.4f"
      % (np.sqrt((unsorted_residual ** 2).mean()),
         np.corrcoef(unsorted_residual, np.arange(len(unsorted_residual)))[0, 1]))

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.2), sharey=True)

left.scatter(np.arange(len(sorted_residual)), sorted_residual, s=13, alpha=0.6, color="#D55E00")
left.set_title("Rows sorted by stops: a clear U", fontsize=11)
right.scatter(np.arange(len(unsorted_residual)), unsorted_residual, s=13, alpha=0.6,
              color="#0072B2")
right.set_title("Rows in their original order: nothing", fontsize=11)

for ax in (left, right):
    ax.axhline(0, color="#000000", linewidth=1.5)
    ax.set_xlabel("row index")
left.set_ylabel("residual (minutes)")

plt.tight_layout()
plt.show()

**Both fits are identical - RMSE 5.5615 either way**, because least squares does not care what order the
rows arrive in. Only the plots differ.

**Sorted, the row-index plot shows the same ∪ as the `stops` plot**, with a residual-index correlation of
only **-0.0079**, which hides how strong the shape is - a ∪ is symmetric, so a *linear* correlation reads near zero
while the picture screams. Look at the plot, not only at the correlation.

**Unsorted, the row-index plot shows nothing**, correlation -0.0136 - statistically the same
non-answer, and this time an honest one.

**Which one is telling you about the model?** Neither, strictly - and that is the answer. The sorted plot
is showing you the `stops` fault a second time, wearing a disguise, because the row index has become a
stand-in for `stops`. The unsorted plot is showing you that the file's order carries no information,
which is a fact about the *file*.

**Why the plot is still worth making:** in real data you rarely know how the file was ordered. If a
row-index plot shows drift, either the rows are sorted by something meaningful, or there is genuine time
structure - and 04-04 established what that means for splitting. The plot is a cheap check on an
assumption everything else depends on.

### E12 · California residuals against latitude and longitude

In [ ]:
california = fetch_california_housing(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    california.data, california.target, test_size=0.25, random_state=0)
census_model = LinearRegression().fit(X_train, y_train)
test_residual = y_test.to_numpy() - census_model.predict(X_test)

located = pd.DataFrame({"latitude": X_test["Latitude"].to_numpy(),
                        "longitude": X_test["Longitude"].to_numpy(),
                        "residual": test_residual})
located["cell"] = (pd.cut(located.latitude, 20, labels=False).astype(str) + "_"
                   + pd.cut(located.longitude, 20, labels=False).astype(str))
cell_mean = located.groupby("cell")["residual"].transform("mean")

print("total residual variance    %.4f" % located.residual.var())
print("variance of the cell means %.4f  = %.1f%% of the total"
      % (cell_mean.var(), 100 * cell_mean.var() / located.residual.var()))
print("\ncurrent RMSE                          %.4f" % np.sqrt((test_residual ** 2).mean()))
print("RMSE if each cell got its own offset  %.4f"
      % np.sqrt(((located.residual - cell_mean) ** 2).mean()))

populated = located.groupby("cell")["residual"].agg(["size", "mean"])
populated = populated[populated["size"] >= 15]
print("\n%d cells hold at least 15 rows; their mean residuals run %.3f to %.3f (sd %.3f)"
      % (len(populated), populated["mean"].min(), populated["mean"].max(),
         populated["mean"].std()))

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.8, 5.2))

limit = 1.2
scatter = left.scatter(located.longitude, located.latitude, c=located.residual.clip(-limit, limit),
                       cmap="coolwarm", s=7, alpha=0.75, vmin=-limit, vmax=limit)
left.set_xlabel("longitude")
left.set_ylabel("latitude")
left.set_title("Residuals on the map: red under-predicted, blue over", fontsize=11)
fig.colorbar(scatter, ax=left, label="residual (clipped)")

for column, colour, marker in [("latitude", "#0072B2", "o"), ("longitude", "#D55E00", "s")]:
    fifths = pd.qcut(located[column], 5, labels=False)
    means = located.groupby(fifths)["residual"].mean()
    right.plot(range(5), means.to_numpy(), marker + "-", color=colour, linewidth=2,
               markersize=8, label="by %s fifth" % column)
right.axhline(0, color="#000000", linewidth=1.4)
right.set_xticks(range(5))
right.set_xticklabels(["lowest", "2nd", "3rd", "4th", "highest"])
right.set_ylabel("mean residual")
right.set_ylim(-0.6, 0.6)
right.set_title("One feature at a time finds almost nothing", fontsize=11)
right.legend(fontsize=9)

plt.tight_layout()
plt.show()

**The one-feature-at-a-time sweep finds almost nothing, and the map finds a great deal.** That gap is the
lesson of this exercise.

Sliced by latitude alone or longitude alone, the mean residual barely leaves the band ±0.1. Sliced by
**position** - a 20x20 grid - **18.1% of the residual variance is between cells**, and among the 49 cells
holding at least fifteen rows the mean residual runs from **-1.129 to +0.706**. If every cell were given
its own offset the RMSE would fall from **0.7351 to 0.6652**.

**Why the single-feature plots missed it:** the fault is not "the model is wrong in the north". It is
"the model is wrong in *this bay* and right in *that valley*", and averaging over a whole latitude band
mixes both together into zero.

**What feature would I want?** Something that names the neighbourhood: distance to the coast, distance to
the nearest city centre, or the school district or census tract as a category. The residual map is
showing that **location carries information the eight columns do not**, and that latitude and longitude
as two independent linear terms is a poor way to express it - because the effect of latitude depends
entirely on the longitude.

**The honest caveat.** Some of that 18.1% is the model fitting the noise of small cells: a cell with
fifteen rows has a noisy mean, and 0.6652 is what these cells achieve on the rows used to compute their
own offsets, so it flatters itself. The finding that survives is the *pattern* - large contiguous
regions of one sign, visible in the map - not the exact number.

## Interpretation

### E13 · A flat band with an RMSE-to-MAE ratio of 4.1

**Both observations are true and they are about different things.**

The residual plot shows the relationship between the residuals and the prediction: flat means the model
has no *systematic* fault, no curve, no fan, no bands.

The ratio 4.1 on 300 rows - where the ceiling is sqrt(300) = 17.3 - says the residuals are very unequal
in *size*: most are small and a handful are enormous. Those outliers do not have to sit anywhere
particular on the fitted axis, so they leave the band flat while dominating the RMSE.

**What to do:** the 05-04 answer, which the flat plot does not replace. Sort by absolute residual, read
the top twenty rows, and ask what they have in common. A flat plot rules out four faults; it says nothing
about a few rows being broken.

### E14 · A fan, reported to a manager

> **"We are typically 8 minutes out on short jobs and 30 minutes out on long ones, so I am quoting the
> error as a percentage of the job rather than a single number of minutes."**

The sentence does three things: it gives the range rather than an average that describes neither end, it
gives the *reason* the single number was refused, and it hands over a replacement rather than a
complaint. That last part is what makes it repeatable in a meeting you are not in.

**What not to say:** "the residuals are heteroscedastic". It is correct, it is unrepeatable, and it
sounds like a request to stop asking.

## Debugging

### E15 · A tight downward diagonal holding 8% of the rows

**Cause one: the target is capped.** All the rows on the line share one actual value, so
`residual = cap - prediction`, a line of slope -1. This is the California case exactly.

**Cause two: those rows share an imputed or default target** - a placeholder like 0, or a
"unknown, use the average" rule applied upstream. The geometry is identical because the mechanism is
identical: one constant actual across many different predictions.

**How to tell them apart in one line:**

```python
print(y[np.abs(residual + prediction - y) < 1e-9].value_counts().head())
```

or more simply, since both causes mean "these rows share one value":

```python
print(y.value_counts().head())
```

**If the repeated value is the maximum of the target, it is a cap. If it is a round number in the middle
of the range - 0, the mean, a suspicious 999 - it is a placeholder.** The distinction matters: a cap is a
limitation to report, while a placeholder is usually a bug to fix upstream, and it may mean those rows
should not be in the training data at all.

### E16 · The plot went flat but held-out RMSE got worse

**What happened: the squared term fitted noise in the training rows.** The residual plot was drawn on
training data, where extra capacity always improves the fit - which is 05-04's point that R-squared
cannot fall when you add a column, restated. A flat *training* residual plot is not evidence of a good
model, for exactly the reason a high training R-squared is not.

**What to check, in order:**

1. **Draw the residual plot on the held-out rows.** If it shows the fault the training plot no longer
   does, the term is fitting noise.
2. **Was the curve real?** Look at the pre-fix plot again. A shallow bend over 50 rows can be luck; the
   chapter's ∪ over 400 rows is not.
3. **How many rows per parameter?** 05-04's degree sweep collapsed at 7 parameters on 25 rows.
4. **Repeat the split a few times.** 04-03 established that a single test set is a lottery, and a small
   RMSE increase may be inside its noise.

**The habit this is teaching:** diagnostics live under the same rule as metrics. **Say which rows you
computed them on.**

## Exam and interview reasoning

### E17 · "How do you know when a regression model is finished?"

> "When the residuals have no structure left that I can act on. I plot them against the fitted values and
> against every feature, and I group them by columns that are not in the model - customer, region, month
> - because that is where a missing feature shows up. If all of those are flat and the held-out error is
> acceptable to whoever is using it, it is finished. If the plots are flat and the error is still too
> large, the model is finished but the *feature set* is not."

**"What if the residual plot is flat but the error is still too large for the business?"**

> "Then the model has extracted what is in these columns, and the answer is more information rather than
> a better algorithm. I would try a different model class once to confirm - if gradient boosting on the
> same columns does no better, the ceiling is the data. After that the useful conversation is about what
> is not being recorded: the residual plot cannot show me a column I do not have, so I would go and ask
> the people who do the work what they know that the data does not."

**What is being tested** is whether "improve the model" means "try another algorithm" to you, or whether
you can tell the difference between a model that has not extracted the signal and a dataset that does not
contain it. The second question is checking that you will not spend three weeks tuning against a ceiling.

## Transfer to a different situation

### E18 · Length of stay: a fan and a hard edge at the low end

**The hard edge is a data property.** Length of stay cannot be negative, and in practice cannot be much
below one day. When the model predicts 0.5 days for a patient who stayed 1 day, the residual is +0.5; it
can never be very negative for short stays, because there is nothing below the floor to be wrong about.
That produces a boundary in the residual plot with nothing beneath it - the mirror image of the
California ceiling. **It is not a fault to fix in the model.** It is a fact about the target, to be
stated, and possibly handled by predicting `log(days)` so the floor stops being reachable.

**The fan is a modelling problem, and a real one.** Long stays are more variable than short ones -
straightforward admissions are predictable, complicated ones are not - so the spread grows with the
prediction. Two things follow:

1. **A single RMSE is misleading.** It will be dominated by the long stays and will overstate the error
   on the routine cases that are most of the ward.
2. **A log target is worth testing here**, since the multiplicative story is plausible: a 20-day stay
   being 5 days out is comparable to a 2-day stay being half a day out. Expect the chapter's
   trade-off - the diagnostic gets readable, the day-scale spread does not vanish, and the
   back-transformation needs care.

**The judgement being tested** is separating "the world is like this" from "my model is wrong". The floor
is the world. The fan is partly the world and partly a modelling choice - and the difference matters,
because you can spend a long time trying to fix a floor.

## Explain it to someone non-technical

### E19 · Why look at the mistakes when the accuracy is fine?

> The accuracy number is an average, and an average can hide a pattern. Last week a model like this one
> looked fine overall, and it turned out to be wrong in the same direction for every job with more than
> eight stops - which we could fix in an afternoon once we could see it. The number could never have told
> us that; it had already been averaged away. So I want to look at where the mistakes fall, not just how
> big they are on average. If there is nothing there, that is worth knowing too, and it takes an hour.

*(89 words.)* The move is to promise a **specific, cheap, bounded** action with a concrete example, rather
than to defend a principle.

## Optional challenge

### E20 · Two datasets, identical MAE, RMSE and R-squared, opposite faults

**The trick is to control the residuals directly instead of hoping.** If a residual vector `e` satisfies
`sum(e) = 0` and `sum(x * e) = 0`, then fitting `y = 3 + 2x + e` on `x` returns exactly the line
`3 + 2x`, and `e` comes back as the residual unchanged. Projecting any vector off the span of `[1, x]`
forces both conditions.

Once the residual vector is under your control:

- **RMSE and MAE** depend only on the residual values.
- **R-squared** does too, because `var(y) = 4 * var(x) + var(e)` when `e` is orthogonal to `x`, so the
  denominator moves with the numerator.

The cheapest answer is `e` and `-e`: identical everything, a ∪ against a ∩. The interesting answer is two
*different kinds* of fault, which needs one number solved for.

In [ ]:
challenge_rng = np.random.default_rng(77)
n_challenge = 240
cx = np.sort(challenge_rng.uniform(0, 10, n_challenge))
design = np.column_stack([np.ones(n_challenge), cx])
projector = design @ np.linalg.pinv(design)


def orthogonal_to_fit(vector):
    return vector - projector @ vector          # forces sum(e)=0 and sum(x e)=0


cg = challenge_rng.integers(0, 2, n_challenge)
cz = challenge_rng.normal(0, 1, n_challenge)

curvature = orthogonal_to_fit((cx - 5) ** 2)
target_rmse = float(np.sqrt((curvature ** 2).mean()))
target_mae = float(np.abs(curvature).mean())


def banded(noise_sd):
    raw = orthogonal_to_fit(10.0 * (cg - 0.5) + noise_sd * cz)
    return raw * target_rmse / float(np.sqrt((raw ** 2).mean()))   # RMSE matched by scaling


low, high = 0.0, 30.0                            # then bisect the ONE remaining knob for MAE
for _ in range(80):
    middle = (low + high) / 2
    if np.abs(banded(middle)).mean() > target_mae:
        low = middle
    else:
        high = middle
bands = banded((low + high) / 2)

print("solved noise sd = %.4f (the group gap was fixed at 10.0)\n" % ((low + high) / 2))
for label, e in [("curvature", curvature), ("bands", bands)]:
    y = 3 + 2 * cx + e
    model = LinearRegression().fit(cx.reshape(-1, 1), y)
    residual = y - model.predict(cx.reshape(-1, 1))
    print("%-10s RMSE %.6f   MAE %.6f   R2 %.6f   fitted slope %.6f"
          % (label, np.sqrt((residual ** 2).mean()), np.abs(residual).mean(),
             model.score(cx.reshape(-1, 1), y), model.coef_[0]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)

axes[0].scatter(3 + 2 * cx, curvature, s=15, alpha=0.6, color="#D55E00")
axes[0].set_title("Dataset 1: curvature", fontsize=11)

for value, marker, colour in [(0, "o", "#7B3294"), (1, "^", "#E69F00")]:
    keep = cg == value
    axes[1].scatter((3 + 2 * cx)[keep], bands[keep], s=15, alpha=0.7, marker=marker,
                    color=colour, label="group %d" % value)
axes[1].set_title("Dataset 2: two groups", fontsize=11)
axes[1].legend(fontsize=8.5)

axes[2].scatter(3 + 2 * cx, -curvature, s=15, alpha=0.6, color="#0072B2")
axes[2].set_title("Dataset 3: the cheap answer, curvature negated", fontsize=11)

for ax in axes:
    ax.axhline(0, color="#000000", linewidth=1.5)
    ax.set_xlabel("fitted value")
axes[0].set_ylabel("residual")

plt.tight_layout()
plt.show()

In [ ]:
thirds = pd.qcut(cx, 3, labels=["low x", "mid x", "high x"])
print("mean residual by third of x")
print(pd.DataFrame({"curvature": curvature, "bands": bands})
      .groupby(thirds, observed=True).mean().round(3).to_string())
print("\nmean residual by group")
print(pd.DataFrame({"curvature": curvature, "bands": bands})
      .groupby(cg).mean().round(3).rename(index={0: "group 0", 1: "group 1"}).to_string())

**RMSE 7.231024, MAE 6.201088, R-squared 0.399016 - the same to six decimal places in both datasets, and
the faults have nothing in common.** Curvature is caught only by the thirds of x (+3.29, -7.40, +4.11)
and bands only by the group split (-5.42, +6.73); each is invisible in the other's table.

**What this proves:** metrics are a **many-to-one** map. Any number of different failure modes flow into
the same three numbers, and no arithmetic recovers which one you have, because the information was
discarded on the way in.

So "RMSE 7.23, MAE 6.20, R-squared 0.40" is not a description of a model. It is a description of the
*size* of its errors, and the two datasets here prove the size is compatible with faults that call for
completely different actions - a squared term in one case, a missing column in the other.

**The practical rule, which is the whole chapter:** a metric is what you report; a residual plot is what
you decide from. Reporting the metric without ever looking at the plot means choosing an action from
evidence that provably cannot distinguish the options.

### E21 · The intercept is doing the work

In [ ]:
intercept_rng = np.random.default_rng(5)
n_intercept = 240
ix = intercept_rng.uniform(0, 10, n_intercept)
iy = 20 + 2 * ix + intercept_rng.normal(0, 2, n_intercept)   # note the large intercept, 20

for has_intercept in [True, False]:
    model = LinearRegression(fit_intercept=has_intercept).fit(ix.reshape(-1, 1), iy)
    fitted = model.predict(ix.reshape(-1, 1))
    residual = iy - fitted
    print("fit_intercept=%-6s slope %.4f   mean residual %+8.4f   corr(residual, fitted) %+.6f"
          % (has_intercept, model.coef_[0], residual.mean(),
             np.corrcoef(residual, fitted)[0, 1]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3))

for ax, has_intercept, colour in [(axes[0], True, "#009E73"), (axes[1], False, "#D55E00")]:
    model = LinearRegression(fit_intercept=has_intercept).fit(ix.reshape(-1, 1), iy)
    fitted = model.predict(ix.reshape(-1, 1))
    residual = iy - fitted
    ax.scatter(fitted, residual, s=16, alpha=0.6, color=colour)
    ax.axhline(0, color="#000000", linewidth=1.5)
    ax.axhline(residual.mean(), color="#7B3294", linestyle="--", linewidth=1.8,
               label="mean residual %+.2f" % residual.mean())
    ax.set_xlabel("fitted value")
    ax.set_title("fit_intercept=%s   corr = %+.4f"
                 % (has_intercept, np.corrcoef(residual, fitted)[0, 1]), fontsize=11)
    ax.legend(fontsize=9)
axes[0].set_ylabel("residual")

plt.tight_layout()
plt.show()

**With an intercept the correlation is 0.000000. Without one it is -0.978234.**

The reason is that `sum(residual) = 0` is the **first normal equation**, and it exists only because there
is an intercept to differentiate with respect to. Remove the intercept and nothing forces the residuals
to average to zero - here they average **+5.20**, because a line through the origin cannot reach data
whose true intercept is 20, so it compensates with a slope of 5.06 against a true 2.0 and is wrong in a way that varies
systematically with the prediction.

**Two consequences that matter beyond this exercise.**

**The flat reference line is a property of the model, not of residuals in general.** Every claim in this
chapter - "structure you see is real", "the reference is flat" - assumed an intercept. Fit through the
origin, or use a model with no additive constant, and the fitted-value plot needs a different baseline.

**More usefully, this is a diagnostic in its own right.** If a residual plot shows a strong tilt and the
residuals do not average to zero, check whether the model has an intercept before looking for anything
subtler. It is a one-line explanation for a dramatic-looking plot, and `fit_intercept=False` is set by
accident more often than anyone admits.